# Loading dataset and preprocessing

In [2]:
# Load dataset
import pandas as pd
df = pd.read_csv('spam.csv')
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
# Count of emails in each Category
df.groupby('Category').count()

,Message
Category,
ham,4825
spam,747


In [4]:
# Creating spam target column
df['spam'] = df['Category'].apply(lambda x: 1 if x=='spam' else 0)
df.head()

,Category,Message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [5]:
import spacy
nlp = spacy.load('en_core_web_sm', disable=["parser", "tagger", "ner"])

def preprocess(texts):
  results = []
  for doc in nlp.pipe(texts, batch_size=1000):
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    results.append(" ".join(tokens))
  return results

In [6]:
df["preprocessed_txt"] = preprocess(df["Message"])

/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [7]:
# Train test split
from sklearn.model_selection import train_test_split
X = df['preprocessed_txt']
Y = df['spam']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25)

# Applying BoW and naive bayes

In [8]:
# Creating a pipleline which applies CountVectorizer on Message and MultinomialNB algorithm
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

clf = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, Y_train)

Pipeline(steps=[('vectorizer', CountVectorizer()), ('nb', MultinomialNB())])

In [9]:
# Check score
clf.score(X_test, Y_test)

0.9870782483847811

In [10]:
# Predict given emails as spam or not spam
emails = [
    'Hey mohan, can we get together to watch footbal game tomorrow?',
    'Upto 20% discount on parking, exclusive offer just for you. Dont miss this reward!'
]
clf.predict(emails)

array([0, 1])

# Applying n-grams and naive bayes

In [11]:
# Bigrams
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(2,2))),
    ('nb', MultinomialNB())
])

clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

0.9834888729361091

In [12]:
# Unigrams and Bigrams
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('nb', MultinomialNB())
])

clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

0.9877961234745154

In [13]:
# Trigrams
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(3,3))),
    ('nb', MultinomialNB())
])

clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

0.9504666188083274

# Applying Tf-IDF and naive bayes

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
clf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

0.9712849964106246

In [17]:
from sklearn.metrics import classification_report
y_pred = clf.predict(X_test)
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1197
           1       1.00      0.80      0.89       196

    accuracy                           0.97      1393
   macro avg       0.98      0.90      0.93      1393
weighted avg       0.97      0.97      0.97      1393



# Using gensim word2vec

In [21]:
!pip install gensim

In [22]:
import gensim.downloader as api
model = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [24]:
vectors = []
for text in df["Message"]:
    tokens = text.split()
    vec = model.get_mean_vector(tokens)
    vectors.append(vec)

df["vector"] = vectors

In [25]:
df.head(3)

,Category,Message,spam,preprocessed_txt,vector
0,ham,"Go until jurong point, crazy.. Available only ...",0,jurong point crazy available bugis n great wor...,"[0.017343024, 0.01556247, 0.002628992, 0.05181..."
1,ham,Ok lar... Joking wif u oni...,0,ok lar joking wif u oni,"[-0.0421047, 0.028880663, 0.018837307, 0.02708..."
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,free entry 2 wkly comp win fa cup final tkts 2...,"[0.001205553, -0.027647695, -0.023975767, -0.0..."


In [26]:
# Train test split
from sklearn.model_selection import train_test_split
X = df["vector"]
y = df["spam"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [27]:
import numpy as np
X_train = np.stack(X_train)
X_test = np.stack(X_test)

In [28]:
# Applying Naive bayes
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.8663677130044843

In [31]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      1.00      0.93       966
           1       0.00      0.00      0.00       149

    accuracy                           0.87      1115
   macro avg       0.43      0.50      0.46      1115
weighted avg       0.75      0.87      0.80      1115



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [32]:
# Applying gradient boosting classifier
from sklearn.ensemble import GradientBoostingClassifier
clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('gbc', GradientBoostingClassifier())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9730941704035875

In [33]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       0.97      0.83      0.89       149

    accuracy                           0.97      1115
   macro avg       0.97      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115

